# Neural Network Regressor — SHAP-based feature selection

Mirrors `Classification/NN_Classifier/NN_Class_Full_feature_importance.ipynb`, but for the
electron-energy regression task.

Pipeline:
1. Train a `ThreeLayerRegressor` on the full (post-correlation-drop) feature set, on `log(E)`.
2. Compute SHAP values with `shap.GradientExplainer`.
3. Rank features by `mean(|SHAP|)` and persist the top-20 to `Input_lists/NN_REG_INPUT.txt`.
4. Retrain a fresh model on those 20 features and save a self-contained inference artifact.

In [ ]:
import copy
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.append('../..')  # Electron_Project root
from Modules.Utils import XGB_REG_DATALOADER  # filters to true electrons
from Modules.models import ThreeLayerRegressor

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

## Data — full feature set, split, scale

`XGB_REG_DATALOADER` filters to true electrons (`p_Truth_isElectron == 1`),
drops the correlated half of each correlated pair, and returns an 80/20
train/test split. We carve a further val set out of the train portion so we
have train/val/test = 64/16/20 of the electron-only data.

In [ ]:
DATA_PATH = '../../Data/AppML_InitialProject_train.h5'
TARGET_COL = 'p_Truth_Energy'

X_trainval, X_test, y_trainval, y_test = XGB_REG_DATALOADER(
    DATA_PATH, TARGET_COL, test_size=0.2,
)
features = list(X_trainval.columns)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.2, random_state=SEED,
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

# Log-target trick — see XGB_Reg.py: MSE on log(E) ≈ relative error on E.
log_y_train = np.log(y_train.values).astype(np.float32)
log_y_val = np.log(y_val.values).astype(np.float32)
log_y_test = np.log(y_test.values).astype(np.float32)

print(f"Features: {len(features)}")
print(f"Train: {X_train_s.shape}   Val: {X_val_s.shape}   Test: {X_test_s.shape}")
print(f"Target range: [{y_trainval.min():.2f}, {y_trainval.max():.2f}] GeV")

## Training & evaluation helpers

In [ ]:
def make_loader(X, y_log, batch_size, shuffle):
    ds = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y_log, dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def relmad(y_true, y_pred):
    """Project's grading metric: mean(|E_pred − E_true| / E_true)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred - y_true) / y_true))


def predict_geV(model, X_scaled):
    """Forward pass on already-scaled features and undo the log transform."""
    model.eval()
    with torch.no_grad():
        log_pred = model(torch.tensor(X_scaled, dtype=torch.float32).to(device))
    return np.exp(log_pred.cpu().numpy())


def train_one(model, train_loader, val_loader, lr, weight_decay,
              max_epochs=200, patience=15):
    """Train one configuration. Returns best_state_dict + loss curves + best epoch."""
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()

    best_val = float('inf')
    best_state, best_epoch, counter = None, 0, 0
    train_losses, val_losses = [], []

    for epoch in range(1, max_epochs + 1):
        model.train()
        tl = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tl += loss.item()
        tl /= len(train_loader)

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                vl += criterion(model(xb), yb).item()
        vl /= len(val_loader)
        scheduler.step(vl)

        train_losses.append(tl)
        val_losses.append(vl)

        if vl < best_val:
            best_val = vl
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

    return best_state, train_losses, val_losses, best_epoch

## Train on the full feature set

Layer widths bumped relative to the top-20 model because the input has ~5×
more features. These are reasonable defaults — the goal is just to get a
working model whose gradients reflect the data, not to set a leaderboard.

In [ ]:
full_params = {
    'first_layer':  256,
    'second_layer': 128,
    'third_layer':  64,
    'dropout':      0.05,
    'lr':           1e-3,
    'weight_decay': 1e-4,
    'batch_size':   256,
}

train_loader = make_loader(X_train_s, log_y_train, full_params['batch_size'], shuffle=True)
val_loader = make_loader(X_val_s, log_y_val, full_params['batch_size'], shuffle=False)

model = ThreeLayerRegressor(
    input_size=len(features),
    first_layer_size=full_params['first_layer'],
    second_layer_size=full_params['second_layer'],
    third_layer_size=full_params['third_layer'],
    dropout=full_params['dropout'],
).to(device)

best_state, train_losses, val_losses, best_epoch = train_one(
    model, train_loader, val_loader,
    lr=full_params['lr'], weight_decay=full_params['weight_decay'],
    max_epochs=200, patience=20,
)
model.load_state_dict(best_state)

y_pred_val = predict_geV(model, X_val_s)
y_pred_test = predict_geV(model, X_test_s)

print(f"Best epoch: {best_epoch}")
print(f"\n=== full_features (NN) ===")
print(f"  RelMAD (val):  {relmad(y_val.values, y_pred_val):.5f}")
print(f"  RelMAD (test): {relmad(y_test.values, y_pred_test):.5f}")

## Diagnostics — training curves, pred-vs-true, relative error

In [ ]:
PLOT_DIR = Path('NN_Reg_plots')
PLOT_DIR.mkdir(exist_ok=True)
TAG = 'full_features_NN'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, label='Train MSE (log E)')
axes[0].plot(val_losses, label='Val MSE (log E)')
axes[0].axvline(best_epoch - 1, color='gray', ls='--', label=f'best epoch = {best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE on log(E)')
axes[0].set_title('Training curves'); axes[0].legend()

axes[1].scatter(y_test, y_pred_test, alpha=0.2, s=5)
lo = float(min(np.min(y_test), np.min(y_pred_test)))
hi = float(max(np.max(y_test), np.max(y_pred_test)))
axes[1].plot([lo, hi], [lo, hi], 'r--', label='y = x')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('True energy (GeV)'); axes[1].set_ylabel('Predicted energy (GeV)')
axes[1].set_title('Predicted vs true (test)'); axes[1].legend()

rel = (np.asarray(y_pred_test) - np.asarray(y_test)) / np.asarray(y_test)
axes[2].hist(rel, bins=120, range=(-0.5, 0.5))
axes[2].axvline(0, color='red', ls='--')
axes[2].axvline(rel.mean(), color='orange', ls='--', label=f'mean = {rel.mean():+.4f}')
axes[2].axvline(np.median(rel), color='green', ls='--', label=f'median = {np.median(rel):+.4f}')
axes[2].set_xlabel('(E_pred − E_true) / E_true'); axes[2].set_ylabel('Count')
axes[2].set_title('Relative-error distribution (test)'); axes[2].legend()

plt.suptitle(f'{TAG} — RelMAD test = {relmad(y_test.values, y_pred_test):.4f}')
plt.tight_layout()
plt.savefig(PLOT_DIR / f'{TAG}_diagnostics.png', dpi=120)
plt.show()

## SHAP feature importance

SHAP values are in **log-space** (the model predicts `log(E)`), so each value
is a feature's contribution to `log(E_pred)`. Ranking by `mean(|SHAP|)` is
scale-invariant under the log transform, so this is the right quantity to
use for feature selection regardless.

In [ ]:
# ThreeLayerRegressor.forward squeezes to 1-D output; shap.GradientExplainer
# wants (batch, n_outputs). Wrap so the output is (batch, 1).
class _ShapWrapper(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, x):
        y = self.base(x)
        return y.unsqueeze(-1) if y.ndim == 1 else y

model_cpu = _ShapWrapper(model.to('cpu').eval()).eval()

rng = np.random.default_rng(SEED)
bg_idx = rng.choice(len(X_train_s), size=min(200, len(X_train_s)), replace=False)
explain_idx = rng.choice(len(X_test_s), size=min(2000, len(X_test_s)), replace=False)

background = torch.tensor(X_train_s[bg_idx], dtype=torch.float32)
X_explain = torch.tensor(X_test_s[explain_idx], dtype=torch.float32)

explainer = shap.GradientExplainer(model_cpu, background)
shap_values = explainer.shap_values(X_explain)

# GradientExplainer returns either an array or a list (per output). Normalise
# to (n_samples, n_features).
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values).squeeze()

shap.summary_plot(
    shap_values, features=X_test_s[explain_idx], feature_names=features,
    show=True,
)


In [ ]:
TOP_N = 20
FEATURE_LIST_OUT = '../Input_lists/NN_REG_INPUT.txt'

feature_importance = np.abs(shap_values).mean(axis=0)
ranking = sorted(zip(feature_importance, features), reverse=True)
top_features = [f for _, f in ranking[:TOP_N]]

with open(FEATURE_LIST_OUT, 'w') as f:
    for feat in top_features:
        f.write(f"{feat}\n")

print(f"Top {TOP_N} features by mean(|SHAP|):")
for i, (imp, feat) in enumerate(ranking[:TOP_N], 1):
    print(f"  {i:>2}. {feat:<45} {imp:.4f}")
print(f"\nWrote → {Path(FEATURE_LIST_OUT).resolve()}")

# Move the trained full-feature model back to device for any later use.
model = model.to(device)

## Optuna tuning on the SHAP-selected 20 features

Sweeps architecture (layer widths, dropout) and optimisation (lr, weight
decay, batch size). Each trial trains a fresh model with early-stopping on
val; the trial value is **RelMAD on val** — the metric the grader uses.
Best params are persisted to `saved_models/NN_Reg_SHAP_params.txt` and reused
by the cell below.


In [ ]:
import optuna

SAVED_DIR = Path('saved_models')
SAVED_DIR.mkdir(exist_ok=True)
PARAMS_PATH = SAVED_DIR / 'NN_Reg_SHAP_params.txt'

N_TRIALS = 400  # match the budget used in NN_Reg.ipynb; lower for quick iteration

# Scale the top-20 subset once — Optuna trials and the final fit reuse these.
scaler_top = StandardScaler().fit(X_train[top_features])
X_train_top_s = scaler_top.transform(X_train[top_features]).astype(np.float32)
X_val_top_s = scaler_top.transform(X_val[top_features]).astype(np.float32)
X_test_top_s = scaler_top.transform(X_test[top_features]).astype(np.float32)


def objective(trial):
    p = {
        'first_layer':  trial.suggest_categorical('first_layer',  [64, 128, 256, 512]),
        'second_layer': trial.suggest_categorical('second_layer', [32, 64, 128, 256]),
        'third_layer':  trial.suggest_categorical('third_layer',  [16, 32, 64, 128]),
        'dropout':      trial.suggest_float('dropout', 0.0, 0.4),
        'lr':           trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True),
        'batch_size':   trial.suggest_categorical('batch_size', [64, 128, 256, 512]),
    }
    tl = make_loader(X_train_top_s, log_y_train, p['batch_size'], shuffle=True)
    vl = make_loader(X_val_top_s, log_y_val, p['batch_size'], shuffle=False)

    m = ThreeLayerRegressor(
        input_size=len(top_features),
        first_layer_size=p['first_layer'],
        second_layer_size=p['second_layer'],
        third_layer_size=p['third_layer'],
        dropout=p['dropout'],
    ).to(device)
    best_state, *_ = train_one(m, tl, vl, lr=p['lr'], weight_decay=p['weight_decay'])
    m.load_state_dict(best_state)
    return relmad(y_val.values, predict_geV(m, X_val_top_s))


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params

with open(PARAMS_PATH, 'w') as f:
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")

print(f"\nBest trial #{study.best_trial.number}: RelMAD = {study.best_value:.5f}")
print("Best params:")
for k, v in best_params.items():
    print(f"  {k:<14} {v}")
print(f"\nSaved → {PARAMS_PATH.resolve()}")


## Final NN trained on the 20 SHAP-selected features

Reuses `best_params` from the Optuna study above and the `scaler_top` /
`X_*_top_s` arrays it produced. Trained with a more generous epoch / patience
budget than each Optuna trial got, so the final fit isn't stopped early by an
unlucky plateau.


In [ ]:
train_loader = make_loader(X_train_top_s, log_y_train, best_params['batch_size'], shuffle=True)
val_loader = make_loader(X_val_top_s, log_y_val, best_params['batch_size'], shuffle=False)

model_top = ThreeLayerRegressor(
    input_size=len(top_features),
    first_layer_size=best_params['first_layer'],
    second_layer_size=best_params['second_layer'],
    third_layer_size=best_params['third_layer'],
    dropout=best_params['dropout'],
).to(device)

best_state, train_losses, val_losses, best_epoch = train_one(
    model_top, train_loader, val_loader,
    lr=best_params['lr'], weight_decay=best_params['weight_decay'],
    max_epochs=300, patience=25,
)
model_top.load_state_dict(best_state)

y_pred_val = predict_geV(model_top, X_val_top_s)
y_pred_test = predict_geV(model_top, X_test_top_s)

print(f"Best epoch: {best_epoch}")
print(f"\n=== top{TOP_N}_SHAP (NN) ===")
print(f"  RelMAD (val):  {relmad(y_val.values, y_pred_val):.5f}")
print(f"  RelMAD (test): {relmad(y_test.values, y_pred_test):.5f}")


In [ ]:
TAG = f'top{TOP_N}_SHAP_NN'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(train_losses, label='Train MSE (log E)')
axes[0].plot(val_losses, label='Val MSE (log E)')
axes[0].axvline(best_epoch - 1, color='gray', ls='--', label=f'best epoch = {best_epoch}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE on log(E)')
axes[0].set_title('Training curves'); axes[0].legend()

axes[1].scatter(y_test, y_pred_test, alpha=0.2, s=5)
lo = float(min(np.min(y_test), np.min(y_pred_test)))
hi = float(max(np.max(y_test), np.max(y_pred_test)))
axes[1].plot([lo, hi], [lo, hi], 'r--', label='y = x')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('True energy (GeV)'); axes[1].set_ylabel('Predicted energy (GeV)')
axes[1].set_title('Predicted vs true (test)'); axes[1].legend()

rel = (np.asarray(y_pred_test) - np.asarray(y_test)) / np.asarray(y_test)
axes[2].hist(rel, bins=120, range=(-0.5, 0.5))
axes[2].axvline(0, color='red', ls='--')
axes[2].axvline(rel.mean(), color='orange', ls='--', label=f'mean = {rel.mean():+.4f}')
axes[2].axvline(np.median(rel), color='green', ls='--', label=f'median = {np.median(rel):+.4f}')
axes[2].set_xlabel('(E_pred − E_true) / E_true'); axes[2].set_ylabel('Count')
axes[2].set_title('Relative-error distribution (test)'); axes[2].legend()

plt.suptitle(f'{TAG} — RelMAD test = {relmad(y_test.values, y_pred_test):.4f}')
plt.tight_layout()
plt.savefig(PLOT_DIR / f'{TAG}_diagnostics.png', dpi=120)
plt.show()

## Save artifact for inference

Same single-`.pth` bundle convention as `NN_Reg.ipynb`: weights, scaler,
feature ordering, hyperparameters, and the log-target flag in one file.

In [ ]:
ARTIFACT_PATH = SAVED_DIR / 'NN_Reg_SHAP_artifact.pth'

torch.save({
    'state_dict': model_top.state_dict(),
    'scaler': scaler_top,
    'features': top_features,
    'params': best_params,
    'target_col': TARGET_COL,
    'log_target': True,
}, ARTIFACT_PATH)

print(f"Saved artifact → {ARTIFACT_PATH.resolve()}")
